# Работаем с локальной LLM с помощью библиотеки openai

В этом тюториале мы научимся общаться с LLM через т.н. OpenAI API.

Код и документация библиотек семейства _openai_: https://github.com/openai/openai-python

Эту библиотеку можно поставить в виртуальное окружение командой `pip install openai`

Импортируем модули которые нам понадобятся впоследствии:

In [1]:
import logging
from timeit import default_timer as timer

from openai import OpenAI

Подготовим логгер:

In [ ]:
logging.basicConfig(level=logging.INFO)

## Делаем запросы к LLM

Подготовим параметры соединения:

In [2]:
api_key = "EMPTY"
api_base_url = "http://localhost:8000/v1"
model_name = "Qwen/Qwen3-4B-FP8"

Создадим клиента:

In [3]:
client = OpenAI(api_key=api_key, base_url=api_base_url, max_retries=0)
logging.info(f"created OpenAI client")

Подготовим промпты:

In [4]:
user_prompt = "What are some of the places to visit around Barcelona?"

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": user_prompt},
]

Дополнительные параметры:

In [5]:
extra_body = {}
extra_body["chat_template_kwargs"] = { "enable_thinking": True }

Подадим запрос к модели, используя _completions API_:

In [6]:
start = timer()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    extra_body=extra_body,
    seed=22,
    #max_tokens=1000,
)
elapsed = timer() - start
print(response)

ChatCompletion(id='chatcmpl-943feaef65463bcc', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\nBarcelona and its surrounding areas offer a wealth of cultural, historical, and natural attractions. Here’s a curated list of must-visit places:\n\n### **In Barcelona**  \n1. **Sagrada Família**  \n   - A masterpiece of Antoni Gaudí, this iconic basilica is a symbol of the city.  \n2. **Park Güell**  \n   - A colorful public park with unique architecture by Gaudí, perfect for a stroll.  \n3. **La Rambla**  \n   - A bustling street lined with shops, street performers, and the Picasso Museum.  \n4. **Gothic Quarter**  \n   - A historic district with medieval architecture, narrow streets, and the Barcelona Cathedral.  \n5. **Camp Nou**  \n   - FC Barcelona’s home stadium, a must-see for football fans.  \n6. **Barceloneta Beach**  \n   - A lively coastal area with beaches, markets, and the famous Barceloneta Night Market.  \n\n---\n\n### **

Распакуем ответ:

In [7]:
# Extract reasoning and content
completion_message = response.choices[0].message
reasoning_content = completion_message.reasoning
content = completion_message.content

# Token usage
num_input_tokens = response.usage.prompt_tokens
num_output_tokens = response.usage.completion_tokens
tps = 1. / (elapsed / num_output_tokens)

# Show results
print("PROMPT:\n")
print(user_prompt)
print("REASONING:\n") 
print(reasoning_content)
print("CONTENT:\n")
print(content)
logging.info(f"got response: num_input_tokens = {num_input_tokens} num_output_tokens = {num_output_tokens}"
             f" tps = {tps:.2f} elapsed = {elapsed:.3f}")

PROMPT:

What are some of the places to visit around Barcelona?
REASONING:


Okay, the user is asking about places to visit around Barcelona. Let me start by recalling the main attractions in and around the city. First, Barcelona itself is a major tourist spot, so I should mention the iconic landmarks like the Sagrada Família, Park Güell, and La Rambla. Then, there are the nearby areas. For example, Girona is a city about an hour away, known for its medieval architecture. The Costa Brava is a coastal area, so maybe include some beaches like Cap de Creus or Tossa de Mar. Also, the Montserrat mountain range is a short drive away, with the Montserrat Monastery and the view of the mountain. The countryside around Tarragona or the coast of the Mediterranean could be other options. I should also think about other nearby cities like Castelldefels or Viladecans, which have cultural sites. Maybe the Barceloneta Beach area is worth mentioning as a nearby resort. Oh, and the city of Girona is a g

Таким образом, мы научились подавать запросы к модели используя библиотеку _openai_.